In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [6]:
X.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,SkewRB,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,0.6019,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,0.4134,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,0.9183,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,1.8028,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,0.8865,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666


In [7]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [8]:
from sklearn.preprocessing import StandardScaler , LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test , y_train, y_test = train_test_split(
    X,y, test_size=0.2 , random_state = 42
    )

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [12]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [13]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [14]:
#Build our Mode

In [15]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self).__init__()

    self.model = nn.Sequential(

        #nn.Linear(input feature , output feature)
        nn.Linear(X_train.shape[1],64),
        nn.ReLU(),
        nn.Linear(64,64),
        nn.ReLU(),
        nn.Linear(64,7)
    )

  def forward(self,x):
      return self.model(x)

In [17]:
model = ANN()
# loss and optim

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [20]:
# Train the NN

epochs = 100
for epoch in range (epochs):
  model.train()

  running_loss = 0.0

  for xb,yb in train_loader:
    optimizer.zero_grad()

    outputs = model(xb)
    loss = criteria(outputs,yb)
    loss.backward()
    optimizer.step()# param update

    running_loss += loss.item()

    train_loss = running_loss/ len(train_loader)

    print(f"epoch = {epoch}/{epochs}, loss = {train_loss}")

epoch = 0/100, loss = 0.08335727194081181
epoch = 0/100, loss = 0.16516216941501782
epoch = 0/100, loss = 0.245264426521633
epoch = 0/100, loss = 0.32487925239231275
epoch = 0/100, loss = 0.4020134728887807
epoch = 0/100, loss = 0.47850899592689844
epoch = 0/100, loss = 0.5565483310948247
epoch = 0/100, loss = 0.6317365014034769
epoch = 0/100, loss = 0.7055224594862565
epoch = 0/100, loss = 0.78073091092317
epoch = 0/100, loss = 0.8553719313248344
epoch = 0/100, loss = 0.9292267146317855
epoch = 0/100, loss = 1.0008864195450493
epoch = 0/100, loss = 1.067815671796384
epoch = 0/100, loss = 1.1366536098977793
epoch = 0/100, loss = 1.2076154066168743
epoch = 0/100, loss = 1.2773464710816094
epoch = 0/100, loss = 1.338657462078592
epoch = 0/100, loss = 1.400035516075466
epoch = 0/100, loss = 1.4589657472527546
epoch = 0/100, loss = 1.5196128202521282
epoch = 0/100, loss = 1.5819144922754038
epoch = 0/100, loss = 1.6428355704183164
epoch = 1/100, loss = 0.06003041371055271
epoch = 1/100, lo

In [23]:
#Evaluate

model.eval()

total = 0
correct = 0

with torch.no_grad():
  for xb,yb in test_loader:
    outputs = model(xb)  # [0.2,0.5,1.3,-0.5...] -7 vals
    _, predicted = torch.max(outputs,1)

    correct += (predicted == yb).sum().item()
    total += yb.size(0) # actual samples in each batch


print("total vals : ",total)
print("correct vals : ",correct)
print("accuracy : ",correct/total  * 100)

total vals :  180
correct vals :  170
accuracy :  94.44444444444444


In [22]:
y_test.shape

(180,)